In [ ]:
import numpy as np
import matplotlib.pyplot as plt

V_rest = -65
V_thresh = -50
RC = 20
dt = 1
I = 16
ref = 0

V = np.ones(500) * V_rest

for t in range(1, len(V)):
    if ref > 0:
        ref = ref - 1
        continue
    V[t] = V[t-1] + dt/RC * (-(V[t-1] - V_rest) + I)
    if V[t] > V_thresh:
        V[t] = V_rest;
        ref = 5 # the neuron rests for 5ms before it can take any new inputs

plt.plot(V)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 1
w = 300.0 # if neuron_0 is fired, then neuron_1 will receive an additional current w
w_list = [None] * 500
A_plus = 20
A_minus = 20
tau_stdp = 20.0

class Neuron:
    def __init__(self, V_rest, V_thresh, RC, I, ref, last_spike_time):
        self.V_rest = V_rest
        self.V_thresh = V_thresh
        self.RC = RC
        self.I = I
        self.ref = ref
        self.status = np.ones(500) * V_rest
        self.last_spike_time = last_spike_time

n_0 = Neuron(-65, -50, 20, 16, 0, -1000) # init last_spike_time to -1000 to represent this neuron has never fired at the beginning
n_1 = Neuron(-65, -50, 20, 0, 0, -1000)

pending_current = 0

for t in range(1, len(n_0.status)):
    
    # after n_0 is fired, n_1 receives an addtional current w in the next timestamp
    if pending_current > 0:
        n_1.I = pending_current
        pending_current = 0
    
    if n_0.ref > 0:
        n_0.ref -= 1
    else:
        n_0.status[t] = n_0.status[t-1] + dt/n_0.RC * (-(n_0.status[t-1] - n_0.V_rest) + n_0.I)
        if n_0.status[t] > n_0.V_thresh:
            n_0.last_spike_time = t # update its last_spike_time to t
            n_0.status[t] = n_0.V_rest
            n_0.ref = 5
            pending_current = w
            dt_stdp = n_1.last_spike_time - n_0.last_spike_time
            if dt_stdp < 0:  # post before pre
                w -= A_minus * np.exp(dt_stdp / tau_stdp)
            
    if n_1.ref > 0:
        n_1.ref -= 1
    else:
        n_1.status[t] = n_1.status[t-1] + dt/n_1.RC * (-(n_1.status[t-1] - n_1.V_rest) + n_1.I)
        n_1.I = 0
        if n_1.status[t] > n_1.V_thresh:
            print(f"n_1 fired at t={t}") # it's tricky to see it reflected on the graph since firing and resetting happen simultaneously  
            n_1.last_spike_time = t
            n_1.status[t] = n_1.V_rest
            n_1.ref = 5
            dt_stdp = n_1.last_spike_time - n_0.last_spike_time
            if dt_stdp > 0:  # pre before post
                w += A_plus * np.exp(-dt_stdp / tau_stdp)

    w_list[t] = w
    

# plt.plot(n_0.status)
# plt.plot(n_1.status)
# plt.plot(w_list)
fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.plot(n_0.status)
ax1.plot(n_1.status)
ax2.plot(w_list)
plt.show()